In [ ]:
import zodipy 

wavelength = np.array([0.5, 0.55, 0.6, 0.65, 0.7])*u.micron
weights = np.array([0.1, 0.5, 1, 0.5, 0.1])

# Initialize a zodiacal light model at a wavelength/frequency or over a bandpass
model = zodipy.Model(wavelength, weights=weights, extrapolate=True)

# Use Astropy's `SkyCoord` object to specify coordinates
ra = np.random.normal(200,1,100) * u.deg
dec = np.random.normal(45,1,100) * u.deg
obstimes = Time("2025-01-01 00:01:00")
skycoord = SkyCoord(ra, dec, obstime=obstimes, frame="icrs")

# Evaluate the zodiacal light model from Earth
emission = model.evaluate(skycoord, obspos="earth")
print(emission)

# Evaluate the zodiacal light model from somewhere in the 1 AU orbit in Heliocentric cartesian coordinates.  
emission = model.evaluate(skycoord, obspos=np.array([1,1,0])*u.AU)
print(emission)

# Trying to use parallel processing. (Exits with error)
import multiprocessing 
emission = model.evaluate(skycoord, obspos=np.array([1,1,0])*u.AU, nprocesses=multiprocessing.cpu_count())
print(emission)



In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord  # High-level coordinates

import zodipy
import numpy as np

import rosalia as rs
# Initialize a zodiacal light model at a wavelength/frequency or over a bandpass
model = zodipy.Model(np.array([0.59,0.6,0.61])*u.micron, weights=np.array([0.1,1,0.1]), extrapolate=True)

# Use Astropy's `SkyCoord` object to specify coordinates
ra = np.random.normal(200,1,100) * u.deg
dec = np.random.normal(45,1,100) * u.deg
obstimes = Time(["2025-01-01 00:01:00"])
skycoord = SkyCoord(ra, dec, obstime=obstimes, frame="icrs")


# Evaluate the zodiacal light model
emission = model.evaluate(skycoord, obspos=np.array([5,5,0])*u.AU)
print(emission)

emission = model.evaluate(skycoord, obspos="earth")

print(emission)
#> [27.52410841 27.66572294 27.81251906] MJy / sr

In [ ]:
exposure_identity = rs.utils.exposure_inspector("RST_WFI_ROSALIA_ra200_dec-23_test_SCAwfi01.asdf")

In [ ]:
obspos = np.array([exposure_identity["XYZ_HELIO_POS"][0][0].value,
                   exposure_identity["XYZ_HELIO_POS"][1][0].value,
                   exposure_identity["XYZ_HELIO_POS"][2][0].value])*u.AU

In [ ]:
(len(skycoord)*[obspos.value]*u.AU).ndim

In [ ]:
#location = rs.telescopes.Roman.get_location(mjd=obstimes.mjd, pov="@sun")
import multiprocessing 
emission = model.evaluate(skycoord, obspos=len(skycoord)*[obspos.value]*u.AU, nprocesses=5)
print(emission)


In [ ]:

import astropy.units as u
import astropy_healpix as ahp
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time

import zodipy




In [ ]:
import matplotlib.pyplot as plt
ra = np.linspace(0,360,360)
dec = np.linspace(-90,90,180)
wavelength = np.array([1.3,1.4,1.5,1.6,1.7,1.8])
weights = np.array([0.5,0.7,0.8,0.9,0.8,0.5])
expstart = 61041.0


rara, decdec = np.meshgrid(ra,dec)
if True:
    plt.imshow(rara)
    plt.colorbar()
    plt.show()
    plt.imshow(decdec)
    plt.colorbar()
    plt.show()

zodi_output = rs.sky.zodipy_zody(ra=rara.flatten(), dec=decdec.flatten(), wavelength=wavelength, weights=weights, expstart=expstart)
zodi_output = np.reshape(zodi_output, (180,360))
plt.figure(figsize=(16,10))
plt.imshow(-2.5*np.log10(rs.utils.MJysr_to_jyarcsec2(zodi_output.value))+8.9)
#plt.scatter(200.34689352, -23.144416)
plt.colorbar()
plt.show()

In [ ]:
import rosalia as rs
zodi_model = rs.sky.zodipy_zody(ra=rara.flatten(), dec=decdec.flatten(), wavelength=wavelength, weights=weights, expstart=expstart)


In [ ]:

model = zodipy.Model(30 * u.micron)

healpix = ahp.HEALPix(nside=256, frame="galactic")
pixels = np.arange(healpix.npix)
skycoord = healpix.healpix_to_skycoord(pixels)

# Note that we manually set the obstime attribute
skycoord.obstime = Time("2022-01-14")

emission = model.evaluate(skycoord, nprocesses=multiprocessing.cpu_count())

# Plot with healpy
hp.mollview(
    emission,
    unit="MJy/sr",
    cmap="afmhot",
    min=0,
    max=80,
    title="Zodiacal light at 30 µm (2022-01-14)",
)
# plt.savefig("../img/healpix_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
skycoord